# 01 Load and QC Datalogger Data

This notebook loads ESP8266 datalogger measurement CSV files from **2026-05-29 onward only**. It preserves duplicated sensor columns from the two I2C buses by renaming repeated headers with `__bus0`, `__bus1`, ... suffixes.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import csv
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

DATA_START_DATE = "20260615"
ANALYSIS_TIMEZONE = "Europe/Zurich"
ANALYSIS_RUN_DATE = pd.Timestamp(datetime.now(ZoneInfo(ANALYSIS_TIMEZONE)).date())
ANALYSIS_END_DATE = ANALYSIS_RUN_DATE.strftime("%Y%m%d")
ANALYSIS_MAX_TIME_EXCLUSIVE = ANALYSIS_RUN_DATE + pd.Timedelta(days=1)
EXPECTED_INTERVAL_SECONDS = 20
GPS_STALE_THRESHOLD_MS = 120000
DATA_DIR = Path("..") / "test_data"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print("pandas", pd.__version__)
print("data directory", DATA_DIR.resolve())
print("analysis date cutoff", ANALYSIS_RUN_DATE.date())

pandas 3.0.3
data directory C:\Users\jonmuell\Documents\GitHub\datalogger_esp8266\test_data
analysis date cutoff 2026-06-29


In [2]:
def data_file_date(path: Path) -> str:
    match = re.fullmatch(r"data_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def measurement_files(data_dir: Path = DATA_DIR, start_date: str = DATA_START_DATE, end_date: str = ANALYSIS_END_DATE) -> list[Path]:
    files = []
    for path in sorted(data_dir.glob("data_*.csv")):
        date = data_file_date(path)
        if date and start_date <= date <= end_date:
            files.append(path)
    return files


def future_measurement_files(data_dir: Path = DATA_DIR, end_date: str = ANALYSIS_END_DATE) -> list[Path]:
    return sorted(path for path in data_dir.glob("data_*.csv") if data_file_date(path) and data_file_date(path) > end_date)


def csv_data_row_count(path: Path) -> int:
    with path.open("r", encoding="utf-8", errors="replace") as f:
        return max(sum(1 for _ in f) - 1, 0)


def unique_headers(headers: list[str]) -> list[str]:
    total = Counter(headers)
    seen = defaultdict(int)
    out = []
    for header in headers:
        if total[header] == 1:
            out.append(header)
        else:
            bus = seen[header]
            out.append(f"{header}__bus{bus}")
            seen[header] += 1
    return out


def inspect_row_widths(path: Path) -> tuple[list[str], pd.DataFrame]:
    issues = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        raw_headers = next(reader)
        expected_width = len(raw_headers)
        for line_number, row in enumerate(reader, start=2):
            if len(row) != expected_width:
                issues.append({
                    "source_file": path.name,
                    "line_number": line_number,
                    "expected_columns": expected_width,
                    "actual_columns": len(row),
                })
    return raw_headers, pd.DataFrame(issues)


def read_measurement_file(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_headers, width_issues = inspect_row_widths(path)
    names = unique_headers(raw_headers)
    df = pd.read_csv(
        path,
        names=names,
        skiprows=1,
        na_values=["", "nan", "NaN", "NAN"],
        keep_default_na=True,
        on_bad_lines="skip",
    )
    df.insert(0, "source_file", path.name)
    df.insert(1, "source_date", data_file_date(path))
    return df, width_issues


files = measurement_files()
future_files_excluded = future_measurement_files()
future_files_excluded_summary = pd.DataFrame([
    {"source_file": path.name, "source_date": data_file_date(path), "rows": csv_data_row_count(path), "first_time": pd.NaT, "last_time": pd.NaT}
    for path in future_files_excluded
])
if future_files_excluded_summary.empty:
    future_files_excluded_summary = pd.DataFrame(columns=["source_file", "source_date", "rows", "first_time", "last_time"])
assert files, "No measurement files found from the configured start date."
print("Selected measurement files:")
for file in files:
    print(" -", file.name)
if not future_files_excluded_summary.empty:
    print("Future-dated measurement files excluded before loading:")
    display(future_files_excluded_summary)

assert all(DATA_START_DATE <= data_file_date(file) <= ANALYSIS_END_DATE for file in files)

Selected measurement files:
 - data_20260615.csv
 - data_20260616.csv
 - data_20260617.csv
 - data_20260618.csv
 - data_20260619.csv
 - data_20260620.csv
 - data_20260621.csv
 - data_20260622.csv
 - data_20260623.csv
 - data_20260624.csv
 - data_20260625.csv
 - data_20260626.csv
Future-dated measurement files excluded before loading:


,source_file,source_date,rows,first_time,last_time
0,data_20260630.csv,20260630,1,NaT,NaT
1,data_20461231.csv,20461231,1,NaT,NaT


In [3]:
frames = []
width_issue_frames = []
for file in files:
    frame, issues = read_measurement_file(file)
    frames.append(frame)
    width_issue_frames.append(issues)

data = pd.concat(frames, ignore_index=True)
row_width_issues = pd.concat(width_issue_frames, ignore_index=True) if width_issue_frames else pd.DataFrame()

data["timestamp_utc"] = pd.to_datetime(data["timestamp_utc"], errors="coerce", utc=False)
if "timestamp_calc_utc" in data.columns:
    data["timestamp_calc_utc"] = pd.to_datetime(data["timestamp_calc_utc"], errors="coerce", utc=False)
else:
    data["timestamp_calc_utc"] = pd.NaT
if "gps_time_fresh" in data.columns:
    data["gps_time_fresh_bool"] = pd.to_numeric(data["gps_time_fresh"], errors="coerce").fillna(1).astype(bool)
else:
    data["gps_time_fresh_bool"] = True
gps_age = pd.to_numeric(data["gps_age_ms"], errors="coerce") if "gps_age_ms" in data.columns else pd.Series(pd.NA, index=data.index, dtype="Float64")
data["gps_age_too_high"] = gps_age > GPS_STALE_THRESHOLD_MS
data["gps_time_stale"] = (~data["gps_time_fresh_bool"]) | data["gps_age_too_high"].fillna(False)
if "timestamp_calc_source" in data.columns:
    calc_source = data["timestamp_calc_source"].astype("string").str.lower()
else:
    calc_source = pd.Series(pd.NA, index=data.index, dtype="string")
if "timestamp_boot_ms" in data.columns:
    boot_ms = pd.to_numeric(data["timestamp_boot_ms"], errors="coerce")
elif "uptime_ms" in data.columns:
    boot_ms = pd.to_numeric(data["uptime_ms"], errors="coerce")
else:
    boot_ms = pd.Series(pd.NA, index=data.index, dtype="Float64")
data["analysis_time"] = data["timestamp_utc"]
groups = data.groupby("boot_id", dropna=False) if "boot_id" in data.columns else [(None, data)]
for _, group in groups:
    fresh = (~group["gps_time_stale"]) & group["timestamp_utc"].notna() & boot_ms.loc[group.index].notna()
    if not fresh.any():
        continue
    base_idx = fresh[fresh].index[-1]
    base_utc = data.at[base_idx, "timestamp_utc"]
    base_boot = boot_ms.loc[base_idx]
    if pd.isna(base_boot):
        continue
    stale_idx = group.index[group["gps_time_stale"] & boot_ms.loc[group.index].notna()]
    data.loc[stale_idx, "analysis_time"] = base_utc + pd.to_timedelta(boot_ms.loc[stale_idx] - base_boot, unit="ms")
data["analysis_time"] = data["timestamp_calc_utc"].combine_first(data["analysis_time"])
data["time_corrected_from_uptime"] = (calc_source == "uptime").fillna(False) | (
    data["gps_time_stale"].fillna(False) & data["analysis_time"].notna() & data["timestamp_utc"].notna() & (data["analysis_time"] != data["timestamp_utc"])
)
data["file_date"] = pd.to_datetime(data["source_date"], format="%Y%m%d", errors="coerce").dt.date
data["file_date"] = pd.to_datetime(data["source_date"], format="%Y%m%d", errors="coerce")
future_row_mask = (
    data["file_date"].gt(ANALYSIS_RUN_DATE)
    | data["analysis_time"].ge(ANALYSIS_MAX_TIME_EXCLUSIVE)
)
future_rows_dropped = data.loc[future_row_mask].copy()
if future_rows_dropped.empty:
    future_rows_dropped_summary = future_files_excluded_summary.copy()
else:
    future_rows_dropped_summary = (
        future_rows_dropped.groupby(["source_file", "source_date"], as_index=False)
        .agg(rows=("source_file", "size"), first_time=("analysis_time", "min"), last_time=("analysis_time", "max"))
    )
    if not future_files_excluded_summary.empty:
        future_rows_dropped_summary = pd.concat([future_rows_dropped_summary, future_files_excluded_summary], ignore_index=True)
print("Future-dated rows dumped:", int(future_rows_dropped_summary["rows"].sum()) if not future_rows_dropped_summary.empty else 0)
display(future_rows_dropped_summary)
data = data.loc[~future_row_mask].copy()
data["file_date"] = data["file_date"].dt.date
data["date"] = data["analysis_time"].dt.date
data["hour"] = data["analysis_time"].dt.hour

print("Rows loaded:", len(data))
print("Columns loaded:", len(data.columns))
print("Timestamp parse failures:", int(data["timestamp_utc"].isna().sum()))
print("Rows with stale GPS time:", int(data["gps_time_stale"].sum()))
print("Rows corrected from uptime:", int(data["time_corrected_from_uptime"].sum()))
print("Row width issues:", len(row_width_issues))

data.head()

Future-dated rows dumped: 2


,source_file,source_date,rows,first_time,last_time
0,data_20260630.csv,20260630,1,NaT,NaT
1,data_20461231.csv,20461231,1,NaT,NaT


Rows loaded: 43267
Columns loaded: 100
Timestamp parse failures: 12285
Rows with stale GPS time: 12285
Rows corrected from uptime: 12285
Row width issues: 1388


,source_file,source_date,boot_id,sample_counter,uptime_ms,timestamp_boot_ms,reset_reason,gps_date_valid,gps_time_fresh,gps_location_valid,gps_location_fresh,gps_chars_processed,gps_age_ms,gps_location_age_ms,gps_stale_count,gps_recovery_count,timestamp_utc,timestamp_calc_utc,timestamp_calc_source,lat,lon,alt,nb_sat,HDOP,bat.mV,bat.perc,bme_T.C__bus0,bme_RH.perc__bus0,bme_P.Pa__bus0,bme_H2O.mmol_mol__bus0,bme_T.C.cal__bus0,bme_T.flag__bus0,bme_P.Pa.cal__bus0,bme_P.flag__bus0,bme_H2O.mmol_mol.cal__bus0,bme_H2O.mmol_mol.flag__bus0,bme_RH.perc.cal__bus0,bme_RH.flag__bus0,scd_T.C__bus0,scd_RH.perc__bus0,scd_CO2.ppm__bus0,scd_CO2.ppm.cal__bus0,scd_CO2.flag__bus0,sen_T.C__bus0,sen_O2.mmol_mol__bus0,sen_raw_V__bus0,sen_O2.mmol_mol.cal__bus0,sen_O2.flag__bus0,bme_T.C__bus1,bme_RH.perc__bus1,bme_P.Pa__bus1,bme_H2O.mmol_mol__bus1,bme_T.C.cal__bus1,bme_T.flag__bus1,bme_P.Pa.cal__bus1,bme_P.flag__bus1,bme_H2O.mmol_mol.cal__bus1,bme_H2O.mmol_mol.flag__bus1,bme_RH.perc.cal__bus1,bme_RH.flag__bus1,scd_T.C__bus1,scd_RH.perc__bus1,scd_CO2.ppm__bus1,scd_CO2.ppm.cal__bus1,scd_CO2.flag__bus1,sen_T.C__bus1,sen_O2.mmol_mol__bus1,sen_raw_V__bus1,sen_O2.mmol_mol.cal__bus1,sen_O2.flag__bus1,free_heap,max_heap_block,min_heap_block,i2c_slow_count,i2c_error_count,i2c_recovery_count,i2c_bus0_error_count,i2c_bus0_consecutive_error_count,i2c_bus0_recovery_count,i2c_bus1_error_count,i2c_bus1_consecutive_error_count,i2c_bus1_recovery_count,write_status,CRC16,gps_poll_timeout_count,gps_i2c_recovery_count,gps_quarantine_active,gps_quarantine_count,valid_sensor_value_count,missing_sensor_value_count,critical_sensor_loss_count,supervisor_reset_count,gps_time_fresh_bool,gps_age_too_high,gps_time_stale,analysis_time,time_corrected_from_uptime,file_date,date,hour
0,data_20260615.csv,20260615,11898506,13428,294158915,294159216,Power On,1,0,1,0,0,14574134,14574101,719,48,NaT,2026-06-15 00:00:02,uptime,NaN,NaN,NaN,14,0.74,2147483647,2147483647,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-1,16792,14192,14192,1,2,98,1,724,49,1,725,49,0,B8D7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,2026-06-15 00:00:02,True,2026-06-15,2026-06-15,0
1,data_20260615.csv,20260615,11898506,13429,294179087,294179388,Power On,1,0,1,0,0,14594306,14594273,720,48,NaT,2026-06-15 00:00:23,uptime,NaN,NaN,NaN,14,0.74,2147483647,2147483647,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-1,16792,14192,14192,1,2,98,1,725,49,1,726,49,0,E057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,2026-06-15 00:00:23,True,2026-06-15,2026-06-15,0
2,data_20260615.csv,20260615,11898506,13430,294199199,294199500,Power On,1,0,1,0,0,14614418,14614385,721,48,NaT,2026-06-15 00:00:43,uptime,NaN,NaN,NaN,14,0.74,2147483647,2147483647,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-1,16792,14192,14192,1,2,98,1,726,49,1,727,49,0,9DE6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,2026-06-15 00:00:43,True,2026-06-15,2026-06-15,0
3,data_20260615.csv,20260615,11898506,13431,294219309,294219610,Power On,1,0,1,0,0,14634528,14634495,722,48,NaT,2026-06-15 00:01:03,uptime,NaN,NaN,NaN,14,0.74,2147483647,2147483647,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,-1,16792,14192,14192,1,2,98,1,727,49,1,728,49,0,AB09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,2026-06-15 00:01:03,True,2026-06-15,2026-06-15,0
4,data_20260615.csv,20260615,11898506,13432,294239421,294239722,Power On,1,0,1,0,0,14654641,14654608,723,49,NaT,2026-06-15 00:01:23,uptime,NaN,NaN,NaN,14,0.74,2147483647,2147483647,NaN,NaN,NaN,NaN,NaN,-1,NaN,-1,NaN,-1,NaN,-3.0,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,-1,Na

In [4]:
def compute_gaps(df: pd.DataFrame, expected_interval_s: int = EXPECTED_INTERVAL_SECONDS) -> pd.DataFrame:
    pieces = []
    valid = df.dropna(subset=["analysis_time"]).sort_values(["source_file", "analysis_time"])
    for source_file, group in valid.groupby("source_file", sort=True):
        ts = group["analysis_time"].reset_index(drop=True)
        deltas = ts.diff().dt.total_seconds()
        gap = pd.DataFrame({
            "source_file": source_file,
            "previous_timestamp": ts.shift(1),
            "analysis_time": ts,
            "delta_seconds": deltas,
        })
        pieces.append(gap)
    out = pd.concat(pieces, ignore_index=True)
    out["missing_samples_est"] = np.maximum(
        0,
        np.floor(out["delta_seconds"].fillna(expected_interval_s) / expected_interval_s).astype(int) - 1,
    )
    return out


gaps_all = compute_gaps(data)
large_gaps = gaps_all[gaps_all["delta_seconds"] > 120].copy()
large_gaps["gap_minutes"] = large_gaps["delta_seconds"] / 60

duplicate_timestamps = (
    data.dropna(subset=["analysis_time"])
    .groupby(["source_file", "analysis_time"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

file_summary = (
    data.groupby("source_file", as_index=False)
    .agg(
        rows=("analysis_time", "size"),
        parseable_timestamps=("analysis_time", "count"),
        start_analysis_time=("analysis_time", "min"),
        end_analysis_time=("analysis_time", "max"),
        gps_sat_median=("nb_sat", "median"),
        hdop_median=("HDOP", "median"),
    )
)
gap_summary = large_gaps.groupby("source_file", as_index=False).agg(
    gaps_gt_120s=("delta_seconds", "size"),
    max_gap_s=("delta_seconds", "max"),
    missing_samples_est=("missing_samples_est", "sum"),
)
file_summary = file_summary.merge(gap_summary, on="source_file", how="left").fillna({
    "gaps_gt_120s": 0,
    "max_gap_s": 0,
    "missing_samples_est": 0,
})

file_summary

,source_file,rows,parseable_timestamps,start_analysis_time,end_analysis_time,gps_sat_median,hdop_median,gaps_gt_120s,max_gap_s,missing_samples_est
0,data_20260615.csv,4295,4295,2026-06-15 00:00:02,2026-06-15 23:59:59,14.0,0.74,0.0,0.0,0.0
1,data_20260616.csv,4293,4293,2026-06-16 00:00:20,2026-06-16 23:59:57,14.0,0.74,0.0,0.0,0.0
2,data_20260617.csv,4205,4205,2026-06-17 00:00:17,2026-06-17 23:59:58,14.0,0.74,1.0,139.0,5.0
3,data_20260618.csv,1385,1385,2026-06-18 00:00:18,2026-06-18 08:27:24,14.0,0.76,1.0,171.0,7.0
4,data_20260619.csv,3712,3712,2026-06-19 00:00:45,2026-06-19 23:59:47,14.0,0.75,1.0,8729.0,435.0
5,data_20260620.csv,4122,4122,2026-06-20 00:00:24,2026-06-20 23:59:44,14.0,0.73,2.0,135.0,10.0
6,data_20260621.csv,4124,4124,2026-06-21 00:00:13,2026-06-21 23:59:39,14.0,0.74,0.0,0.0,0.0
7,data_20260622.csv,4121,4121,2026-06-22 00:00:07,2026-06-22 23:59:57,14.0,0.75,0.0,0.0,0.0
8,data_20260623.csv,4122,4122,2026-06-23 00:00:24,2026-06-23 23:59:38,15.0,0.71,0.0,0.0,0.0
9,data_20260624.csv,4125,4125,2026-06-24 00:00:07,2026-06-24 23:59:25,15.0,0.72,0.0,0.0,0.0


In [5]:
print("Large gaps over 120 seconds:")
display(large_gaps[["source_file", "previous_timestamp", "analysis_time", "delta_seconds", "gap_minutes", "missing_samples_est"]])

print("Duplicate timestamps:", len(duplicate_timestamps))
display(duplicate_timestamps.head(20))

print("Row width issues:", len(row_width_issues))
display(row_width_issues.head(20))

Large gaps over 120 seconds:


,source_file,previous_timestamp,analysis_time,delta_seconds,gap_minutes,missing_samples_est
11086,data_20260617.csv,2026-06-17 14:00:19,2026-06-17 14:02:38,139.0,2.316667,5
12929,data_20260618.csv,2026-06-18 00:46:34,2026-06-18 00:49:25,171.0,2.850000,7
16931,data_20260619.csv,2026-06-19 15:59:57,2026-06-19 18:25:26,8729.0,145.483333,435
18450,data_20260620.csv,2026-06-20 03:13:46,2026-06-20 03:15:51,125.0,2.083333,5
18479,data_20260620.csv,2026-06-20 03:23:20,2026-06-20 03:25:35,135.0,2.250000,5


Duplicate timestamps: 2870


,source_file,analysis_time,count
11916,data_20260617.csv,2026-06-17 19:07:53,6
14183,data_20260619.csv,2026-06-19 00:04:11,2
14194,data_20260619.csv,2026-06-19 00:08:35,2
14198,data_20260619.csv,2026-06-19 00:10:30,2
14204,data_20260619.csv,2026-06-19 00:12:55,2
14208,data_20260619.csv,2026-06-19 00:14:21,2
14213,data_20260619.csv,2026-06-19 00:16:44,2
14219,data_20260619.csv,2026-06-19 00:19:10,2
14235,data_20260619.csv,2026-06-19 00:25:05,2
14239,data_20260619.csv,2026-06-19 00:26:35,2


Row width issues: 1388


,source_file,line_number,expected_columns,actual_columns
0,data_20260618.csv,1387,82,90
1,data_20260618.csv,1388,82,90
2,data_20260618.csv,1389,82,90
3,data_20260618.csv,1390,82,90
4,data_20260618.csv,1391,82,90
5,data_20260618.csv,1392,82,90
6,data_20260618.csv,1393,82,90
7,data_20260618.csv,1394,82,90
8,data_20260618.csv,1395,82,90
9,data_20260618.csv,1396,82,90


In [6]:
hourly_base = data.dropna(subset=["analysis_time"]).assign(
    date=lambda x: x["analysis_time"].dt.strftime("%Y-%m-%d"),
    hour=lambda x: x["analysis_time"].dt.hour,
)
hourly_coverage = hourly_base.groupby(["date", "hour"], as_index=False).agg(
    rows=("analysis_time", "size"),
    unique_timestamps=("analysis_time", "nunique"),
    stale_gps_rows=("gps_time_stale", "sum"),
    corrected_from_uptime_rows=("time_corrected_from_uptime", "sum"),
    max_gps_age_ms=("gps_age_ms", lambda x: pd.to_numeric(x, errors="coerce").max()),
)
hourly_coverage["duplicate_timestamp_rows"] = hourly_coverage["rows"] - hourly_coverage["unique_timestamps"]
hourly_coverage["expected_rows"] = 3600 / EXPECTED_INTERVAL_SECONDS
hourly_coverage["coverage_fraction"] = (hourly_coverage["unique_timestamps"] / hourly_coverage["expected_rows"]).clip(upper=1)
hourly_coverage["stale_gps_fraction"] = hourly_coverage["stale_gps_rows"] / hourly_coverage["rows"]
hourly_coverage["has_stale_gps"] = hourly_coverage["stale_gps_rows"] > 0

display(hourly_coverage.head(48))
display(hourly_coverage.query("date == '2026-05-31'"))

,date,hour,rows,unique_timestamps,stale_gps_rows,corrected_from_uptime_rows,max_gps_age_ms,duplicate_timestamp_rows,expected_rows,coverage_fraction,stale_gps_fraction,has_stale_gps
0,2026-06-15,0,179,179,179,179,18155261,0,180.0,0.994444,1.0,True
1,2026-06-15,1,179,179,179,179,21756532,0,180.0,0.994444,1.0,True
2,2026-06-15,2,179,179,179,179,25357839,0,180.0,0.994444,1.0,True
3,2026-06-15,3,179,179,179,179,28959223,0,180.0,0.994444,1.0,True
4,2026-06-15,4,179,179,179,179,32560602,0,180.0,0.994444,1.0,True
5,2026-06-15,5,179,179,179,179,36162968,0,180.0,0.994444,1.0,True
6,2026-06-15,6,179,179,179,179,39764368,0,180.0,0.994444,1.0,True
7,2026-06-15,7,179,179,179,179,43365784,0,180.0,0.994444,1.0,True
8,2026-06-15,8,179,179,179,179,46967185,0,180.0,0.994444,1.0,True
9,2026-06-15,9,179,179,179,179,50568606,0,180.0,0.994444,1.0,True


,date,hour,rows,unique_timestamps,stale_gps_rows,corrected_from_uptime_rows,max_gps_age_ms,duplicate_timestamp_rows,expected_rows,coverage_fraction,stale_gps_fraction,has_stale_gps


In [7]:
def status_file_date(path: Path) -> str:
    match = re.fullmatch(r"status_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def read_status_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, na_values=["", "nan", "NaN", "NAN"], keep_default_na=True, on_bad_lines="skip")
    frame.insert(0, "source_file", path.name)
    frame.insert(1, "source_date", status_file_date(path))
    return frame


status_files = sorted(path for path in DATA_DIR.glob("status_*.csv") if status_file_date(path) and status_file_date(path) <= ANALYSIS_END_DATE)
if status_files:
    status_data = pd.concat([read_status_file(path) for path in status_files], ignore_index=True, sort=False)
    print("Status rows loaded:", len(status_data))
    display(status_data.head(50))
else:
    status_data = pd.DataFrame()
    print("No status_YYYYMMDD.csv files found yet; this is expected for pre-patch data.")

new_diag_cols = [
    "boot_id", "sample_counter", "uptime_ms", "timestamp_boot_ms", "reset_reason",
    "timestamp_calc_utc", "timestamp_calc_source",
    "gps_date_valid", "gps_time_fresh", "gps_location_valid", "gps_location_fresh",
    "gps_chars_processed", "gps_age_ms", "gps_location_age_ms", "gps_stale_count", "gps_recovery_count",
    "free_heap", "max_heap_block", "min_heap_block", "i2c_slow_count",
    "i2c_error_count", "i2c_recovery_count", "write_status",
]
present_diag_cols = [col for col in new_diag_cols if col in data.columns]
print("New firmware diagnostic columns present:", present_diag_cols)
if present_diag_cols:
    display(data[present_diag_cols].head())

Status rows loaded: 1706


,source_file,source_date,event,boot_id,sample_counter,uptime_ms,reset_reason,free_heap,max_heap_block,min_heap_block,gps_date_valid,gps_location_valid,related_write_status,card_missing_count,header_open_fail_count,append_open_fail_count,print_fail_count,flush_fail_count,close_fail_count,CRC16,gps_time_fresh,gps_location_fresh,gps_stale_count,gps_recovery_count,i2c_error_count,i2c_recovery_count,gps_poll_timeout_count,gps_i2c_recovery_count,gps_quarantine_active,gps_quarantine_count,valid_sensor_value_count,missing_sensor_value_count,critical_sensor_loss_count,supervisor_reset_count
0,status_20260603.csv,20260603,boot,11925535,1,21507,External System,20928,18288,18288,1,1,0,0,0,0,0,0,0,7DB9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,status_20260603.csv,20260603,boot,11919930,1,22554,Software/System restart,20744,18248,18248,1,1,0,0,0,0,0,0,0,B868,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,status_20260603.csv,20260603,boot,11914710,1,22594,Software/System restart,20928,18288,18288,1,1,0,0,0,0,0,0,0,7AEE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,status_20260603.csv,20260603,boot,11923092,1,22561,Software/System restart,20928,18320,18320,1,1,0,0,0,0,0,0,0,8006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,status_20260603.csv,20260603,boot,11923334,1,22660,Software/System restart,20928,18288,18288,1,1,0,0,0,0,0,0,0,F036,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,status_20260603.csv,20260603,boot,11952236,1,22857,External System,20928,18320,18320,1,1,0,0,0,0,0,0,0,0D95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,status_20260603.csv,20260603,boot,11879593,1,22758,External System,20928,18272,18272,1,1,0,0,0,0,0,0,0,DC3A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,status_20260603.csv,20260603,boot,11879022,1,22853,Power On,20928,18288,18288,1,1,0,0,0,0,0,0,0,8334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,status_20260603.csv,20260603,boot,11890964,1,21769,External System,20928,18256,18256,1,1,0,0,0,0,0,0,0,77CA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,status_20260603.csv,20260603,boot,11890940,1,22599,Software/System restart,20928,18272,18272,1,1,0,0,0,0,0,0,0,C267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


New firmware diagnostic columns present: ['boot_id', 'sample_counter', 'uptime_ms', 'timestamp_boot_ms', 'reset_reason', 'timestamp_calc_utc', 'timestamp_calc_source', 'gps_date_valid', 'gps_time_fresh', 'gps_location_valid', 'gps_location_fresh', 'gps_chars_processed', 'gps_age_ms', 'gps_location_age_ms', 'gps_stale_count', 'gps_recovery_count', 'free_heap', 'max_heap_block', 'min_heap_block', 'i2c_slow_count', 'i2c_error_count', 'i2c_recovery_count', 'write_status']


,boot_id,sample_counter,uptime_ms,timestamp_boot_ms,reset_reason,timestamp_calc_utc,timestamp_calc_source,gps_date_valid,gps_time_fresh,gps_location_valid,gps_location_fresh,gps_chars_processed,gps_age_ms,gps_location_age_ms,gps_stale_count,gps_recovery_count,free_heap,max_heap_block,min_heap_block,i2c_slow_count,i2c_error_count,i2c_recovery_count,write_status
0,11898506,13428,294158915,294159216,Power On,2026-06-15 00:00:02,uptime,1,0,1,0,0,14574134,14574101,719,48,16792,14192,14192,1,2,98,0
1,11898506,13429,294179087,294179388,Power On,2026-06-15 00:00:23,uptime,1,0,1,0,0,14594306,14594273,720,48,16792,14192,14192,1,2,98,0
2,11898506,13430,294199199,294199500,Power On,2026-06-15 00:00:43,uptime,1,0,1,0,0,14614418,14614385,721,48,16792,14192,14192,1,2,98,0
3,11898506,13431,294219309,294219610,Power On,2026-06-15 00:01:03,uptime,1,0,1,0,0,14634528,14634495,722,48,16792,14192,14192,1,2,98,0
4,11898506,13432,294239421,294239722,Power On,2026-06-15 00:01:23,uptime,1,0,1,0,0,14654641,14654608,723,49,16792,14192,14192,1,2,98,0


In [8]:
# Future-date checks and GPS/I2C restart diagnostics.
def status_file_date(path: Path) -> str:
    match = re.fullmatch(r"status_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def read_status_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, na_values=["", "nan", "NaN", "NAN"], keep_default_na=True, on_bad_lines="skip")
    frame.insert(0, "source_file", path.name)
    frame.insert(1, "source_date", status_file_date(path))
    return frame


def real_sensor_columns(df: pd.DataFrame) -> list[str]:
    prefixes = ("bme_", "scd_", "sen_", "Sin.", "Sout.", "mlx_")
    blocked = ("flag", ".cal")
    return [col for col in df.columns if col.startswith(prefixes) and not any(token in col for token in blocked)]


status_files = sorted(path for path in DATA_DIR.glob("status_*.csv") if "20260611" <= status_file_date(path) <= ANALYSIS_END_DATE)
if status_files:
    status_data = pd.concat([read_status_file(path) for path in status_files], ignore_index=True, sort=False)
    for col in status_data.columns:
        if col not in {"source_file", "source_date", "event", "reset_reason", "CRC16", "boot_time_source", "supervisor_restart_reason"}:
            status_data[col] = pd.to_numeric(status_data[col], errors="coerce")
else:
    status_data = pd.DataFrame()

sensor_cols = real_sensor_columns(data)
for col in sensor_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")
if "valid_sensor_value_count" not in data.columns:
    data["valid_sensor_value_count"] = data[sensor_cols].notna().sum(axis=1) if sensor_cols else 0
if "missing_sensor_value_count" not in data.columns:
    data["missing_sensor_value_count"] = data[sensor_cols].isna().sum(axis=1) if sensor_cols else 0

for col in [
    "gps_chars_processed", "gps_age_ms", "gps_time_fresh", "gps_time_sane",
    "gps_quarantine_active", "critical_sensor_loss_count", "gps_i2c_hard_fault_count",
    "supervisor_reset_count", "gps_rejected_time_count",
]:
    if col not in data.columns:
        data[col] = pd.NA
    data[col] = pd.to_numeric(data[col], errors="coerce")

source_dates = pd.to_datetime(data["source_date"], format="%Y%m%d", errors="coerce")
future_file_summary = future_rows_dropped_summary.copy()
print("Future-dated data rows dumped before analysis:")
display(future_file_summary)

classified = data.dropna(subset=["analysis_time"]).copy()
valid_sensor_values = pd.to_numeric(classified["valid_sensor_value_count"], errors="coerce").fillna(0)
gps_chars = pd.to_numeric(classified["gps_chars_processed"], errors="coerce")
gps_age = pd.to_numeric(classified["gps_age_ms"], errors="coerce")
gps_fresh = pd.to_numeric(classified["gps_time_fresh"], errors="coerce").fillna(1)
gps_sane = pd.to_numeric(classified["gps_time_sane"], errors="coerce").fillna(1)
gps_quarantine = pd.to_numeric(classified["gps_quarantine_active"], errors="coerce").fillna(0).astype(bool)
hard_fault_counter = pd.to_numeric(classified["gps_i2c_hard_fault_count"], errors="coerce").fillna(0)
supervisor_counter = pd.to_numeric(classified["supervisor_reset_count"], errors="coerce").fillna(0)
rejected_counter = pd.to_numeric(classified["gps_rejected_time_count"], errors="coerce").fillna(0)

classified["gps_dead"] = (gps_fresh.eq(0) | gps_age.gt(GPS_STALE_THRESHOLD_MS)) & gps_chars.fillna(1).eq(0)
classified["sensor_data_lost"] = valid_sensor_values.eq(0)
classified["gps_time_rejected"] = gps_sane.eq(0) & gps_fresh.eq(1)
classified["supervisor_restart"] = supervisor_counter.gt(0)
classified["gps_i2c_hard_fault"] = classified["sensor_data_lost"] & (gps_quarantine | classified["gps_dead"] | hard_fault_counter.gt(0))
classified["boot_waiting_for_gps"] = classified["sensor_data_lost"] & classified["analysis_time"].isna()
classified["fault_class"] = np.select(
    [
        classified["supervisor_restart"],
        classified["gps_i2c_hard_fault"],
        classified["gps_time_rejected"] | rejected_counter.gt(0),
        classified["sensor_data_lost"],
        classified["gps_dead"],
        gps_fresh.eq(0) | gps_age.gt(GPS_STALE_THRESHOLD_MS),
    ],
    ["supervisor_restart", "gps_i2c_hard_fault", "gps_time_rejected", "sensor_data_lost", "gps_dead", "gps_stale_only"],
    default="normal",
)

fault_summary = (
    classified.assign(date=lambda x: x["analysis_time"].dt.strftime("%Y-%m-%d"))
    .groupby(["date", "fault_class"], as_index=False)
    .agg(
        rows=("fault_class", "size"),
        first_time=("analysis_time", "min"),
        last_time=("analysis_time", "max"),
        min_valid_sensor_values=("valid_sensor_value_count", "min"),
        max_gps_age_ms=("gps_age_ms", "max"),
        max_hard_fault_count=("gps_i2c_hard_fault_count", "max"),
        max_supervisor_reset_count=("supervisor_reset_count", "max"),
    )
)
print("GPS/I2C fault classification summary:")
display(fault_summary)

transitions = classified[[
    "analysis_time", "fault_class", "valid_sensor_value_count",
    "gps_chars_processed", "gps_age_ms", "source_file",
]].copy()
transitions["group"] = transitions["fault_class"].ne(transitions["fault_class"].shift()).cumsum()
fault_intervals = (
    transitions.groupby("group", as_index=False)
    .agg(
        fault_class=("fault_class", "first"),
        start=("analysis_time", "min"),
        end=("analysis_time", "max"),
        rows=("fault_class", "size"),
        min_valid_sensor_values=("valid_sensor_value_count", "min"),
        max_gps_age_ms=("gps_age_ms", "max"),
        first_source_file=("source_file", "first"),
        last_source_file=("source_file", "last"),
    )
)
fault_intervals["duration_minutes"] = (fault_intervals["end"] - fault_intervals["start"]).dt.total_seconds() / 60
fault_intervals = fault_intervals[fault_intervals["fault_class"] != "normal"]
print("Fault intervals:")
display(fault_intervals.sort_values("start"))

if not status_data.empty:
    status_events = status_data[status_data["event"].astype(str).str.contains(
        "boot|gps|sensor_data_lost|supervisor", case=False, na=False
    )].copy()
    print("Important status events:")
    display(status_events)

    restart_events = status_data[status_data["event"].eq("supervisor_restart_pending")].copy()
    boot_events = status_data[status_data["event"].eq("boot")].copy()
    restart_records = []
    for _, restart in restart_events.iterrows():
        later_boots = boot_events[
            (boot_events["source_date"] >= restart["source_date"]) &
            (boot_events["boot_id"] != restart.get("boot_id"))
        ]
        next_boot = later_boots.head(1)
        restart_records.append({
            "restart_source_file": restart["source_file"],
            "restart_sample_counter": restart.get("sample_counter"),
            "restart_uptime_ms": restart.get("uptime_ms"),
            "restart_reason": restart.get("supervisor_restart_reason", ""),
            "next_boot_file": next_boot["source_file"].iloc[0] if len(next_boot) else pd.NA,
            "next_boot_id": next_boot["boot_id"].iloc[0] if len(next_boot) else pd.NA,
            "next_boot_reset_reason": next_boot["reset_reason"].iloc[0] if len(next_boot) else pd.NA,
        })
    restart_summary = pd.DataFrame.from_records(restart_records)
    print("Supervisor restart follow-up:")
    display(restart_summary)
else:
    print("No status data available for restart diagnostics.")


Future-dated data rows dumped before analysis:


,source_file,source_date,rows,first_time,last_time
0,data_20260630.csv,20260630,1,NaT,NaT
1,data_20461231.csv,20461231,1,NaT,NaT


GPS/I2C fault classification summary:


,date,fault_class,rows,first_time,last_time,min_valid_sensor_values,max_gps_age_ms,max_hard_fault_count,max_supervisor_reset_count
0,2026-06-15,gps_i2c_hard_fault,4295,2026-06-15 00:00:02,2026-06-15 23:59:59,NaN,100971136,NaN,NaN
1,2026-06-16,gps_i2c_hard_fault,4293,2026-06-16 00:00:20,2026-06-16 23:59:57,NaN,187368703,NaN,NaN
2,2026-06-17,gps_i2c_hard_fault,3369,2026-06-17 00:00:17,2026-06-17 23:59:58,NaN,237791162,NaN,NaN
3,2026-06-17,sensor_data_lost,836,2026-06-17 14:02:38,2026-06-17 19:07:53,NaN,102468,NaN,NaN
4,2026-06-18,gps_i2c_hard_fault,136,2026-06-18 00:00:18,2026-06-18 00:46:34,NaN,20423666,NaN,NaN
5,2026-06-18,sensor_data_lost,1249,2026-06-18 00:49:25,2026-06-18 08:27:24,NaN,265,NaN,NaN
6,2026-06-19,gps_i2c_hard_fault,76,2026-06-19 15:34:27,2026-06-19 15:59:37,0.0,1637287,NaN,0.0
7,2026-06-19,gps_stale_only,6,2026-06-19 02:16:56,2026-06-19 20:58:23,20.0,41911,NaN,0.0
8,2026-06-19,normal,3625,2026-06-19 00:00:45,2026-06-19 23:59:47,12.0,86193,NaN,0.0
9,2026-06-19,sensor_data_lost,4,2026-06-19 15:34:07,2026-06-19 18:25:28,0.0,107129,NaN,0.0


Fault intervals:


,group,fault_class,start,end,rows,min_valid_sensor_values,max_gps_age_ms,first_source_file,last_source_file,duration_minutes
0,1,gps_i2c_hard_fault,2026-06-15 00:00:02,2026-06-17 14:00:19,11086,NaN,237791162,data_20260615.csv,data_20260617.csv,3720.283333
1,2,sensor_data_lost,2026-06-17 14:02:38,2026-06-17 19:07:53,836,NaN,102468,data_20260617.csv,data_20260617.csv,305.250000
2,3,gps_i2c_hard_fault,2026-06-17 19:08:13,2026-06-18 00:46:34,1007,NaN,20423666,data_20260617.csv,data_20260618.csv,338.350000
3,4,sensor_data_lost,2026-06-18 00:49:25,2026-06-18 08:27:24,1249,NaN,265,data_20260618.csv,data_20260618.csv,457.983333
5,6,gps_stale_only,2026-06-19 02:16:56,2026-06-19 02:17:38,3,20.0,41911,data_20260619.csv,data_20260619.csv,0.700000
7,8,gps_stale_only,2026-06-19 08:19:11,2026-06-19 08:19:11,1,20.0,2,data_20260619.csv,data_20260619.csv,0.000000
9,10,sensor_data_lost,2026-06-19 15:34:07,2026-06-19 15:34:07,1,0.0,107129,data_20260619.csv,data_20260619.csv,0.000000
10,11,gps_i2c_hard_fault,2026-06-19 15:34:27,2026-06-19 15:59:37,76,0.0,1637287,data_20260619.csv,data_20260619.csv,25.166667
11,12,supervisor_restart,2026-06-19 15:59:57,2026-06-19 15:59:57,1,0.0,1657409,data_20260619.csv,data_20260619.csv,0.000000
12,13,sensor_data_lost,2026-06-19 18:25:26,2026-06-19 18:25:28,3,0.0,20264,data_20260619.csv,data_20260619.csv,0.033333


Important status events:


,source_file,source_date,event,boot_id,sample_counter,uptime_ms,reset_reason,free_heap,max_heap_block,min_heap_block,gps_date_valid,gps_location_valid,related_write_status,card_missing_count,header_open_fail_count,append_open_fail_count,print_fail_count,flush_fail_count,close_fail_count,CRC16,gps_time_fresh,gps_location_fresh,gps_stale_count,gps_recovery_count,i2c_error_count,i2c_recovery_count,gps_poll_timeout_count,gps_i2c_recovery_count,gps_quarantine_active,gps_quarantine_count,valid_sensor_value_count,missing_sensor_value_count,critical_sensor_loss_count,supervisor_reset_count
0,status_20260611.csv,20260611,boot,11901737,1,22792,Power On,17344,14736,14736,1,1,0,0,0,0,0,0,0,303D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,status_20260614.csv,20260614,gps_stale,11898506,12712,279752400,Power On,16792,14192,14192,1,1,0,0,0,0,0,0,0,AB9F,0.0,0.0,3.0,1.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1578,status_20260617.csv,20260617,gps_recovered,11898506,24514,517403687,Power On,16752,14192,14192,1,1,0,0,0,0,0,0,0,4197,1.0,1.0,0.0,788.0,2.0,1576.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1582,status_20260617.csv,20260617,gps_stale,11898506,25352,535920545,Power On,16672,13304,13304,1,1,0,0,0,0,0,0,0,F12A,0.0,0.0,3.0,789.0,5.0,1580.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1650,status_20260618.csv,20260618,gps_recovered,11898506,26357,556210007,Power On,16560,13304,13304,1,1,0,0,0,0,0,0,0,F704,1.0,1.0,0.0,856.0,5.0,1714.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1653,status_20260619.csv,20260619,gps_stale,11918449,2755,79650366,Software/System restart,15880,12496,12496,1,1,0,0,0,0,0,0,0,A111,0.0,0.0,3.0,1.0,5.0,4.0,2696.0,1.0,0.0,0.0,0.0,20.0,0.0,0.0
1654,status_20260619.csv,20260619,gps_i2c_stale,11918449,2755,79650413,Software/System restart,15880,12496,12496,1,1,0,0,0,0,0,0,0,F039,0.0,0.0,3.0,1.0,5.0,4.0,2696.0,1.0,0.0,0.0,0.0,20.0,0.0,0.0
1655,status_20260619.csv,20260619,gps_i2c_recovery,11918449,2755,79650459,Software/System restart,15880,12496,12496,1,1,0,0,0,0,0,0,0,995E,0.0,0.0,3.0,1.0,5.0,4.0,2696.0,1.0,0.0,0.0,0.0,20.0,0.0,0.0
1657,status_20260619.csv,20260619,gps_i2c_recovery,11918449,2770,79952426,Software/System restart,15880,12496,12496,1,1,0,0,0,0,0,0,0,2151,0.0,0.0,18.0,2.0,5.0,6.0,2696.0,2.0,0.0,0.0,0.0,20.0,0.0,0.0
1659,status_20260619.csv,20260619,gps_i2c_recovery,11918449,2785,80254400,Software/System restart,15880,12496,12496,1,1,0,0,0,0,0,0,0,88A2,0.0,0.0,33.0,3.0,5.0,8.0,2696.0,3.0,1.0,1.0,0.0,20.0,1.0,0.0


Supervisor restart follow-up:


,restart_source_file,restart_sample_counter,restart_uptime_ms,restart_reason,next_boot_file,next_boot_id,next_boot_reset_reason
0,status_20260619.csv,2829,81140273,,status_20260619.csv,11917989,Software/System restart
1,status_20260625.csv,24937,531145672,,status_20260626.csv,11904844,Software/System restart
